# 02 — Elasticity: naive vs causal

Three estimates of the price elasticity of demand, biased → causal:

1. **naive OLS** `log_q ~ log_p` — the biased foil (price endogeneity flattens it toward 0)
2. **fixed effects** — product + time effects strip time-invariant per-product confounding
3. **IV (2SLS)** — instrument price with lagged own price + competitor prices

The **gap between naive and corrected ε** is the headline: pricing on the wrong number
misjudges how customers respond.

In [ ]:
import sys; sys.path.insert(0, "..")
from src.data import load_prepared
from src.elasticity import naive_ols, fixed_effects, iv_2sls, compare_all

df = load_prepared()
for r in (naive_ols(df), fixed_effects(df), iv_2sls(df)):
    print(r)

In [ ]:
# tidy comparison table (point estimate + 95% CI per method)
compare_all(df).round(3)

## Reading the result

- Naive OLS lands near **−0.13** — apparently *inelastic*, which is implausible for discretionary retail goods.
- Adding product + time fixed effects pulls ε to roughly **−2.7** (clearly elastic): once we compare a
  product to *itself* over time, real price sensitivity appears. The pooled regression was confounded by
  expensive products simply selling in different volumes than cheap ones.
- IV with lag/competitor prices is **weak** here (CI brushes zero) — those instruments don't shift this
  product's price much. Worth a first-stage F / weak-instrument check before trusting it. Honesty point:
  report it as inconclusive rather than cherry-picking.

**Caveats to keep:** constant-elasticity is a functional-form assumption; the markdown flag is a proxy
(no true promo field); 52 products × 20 months is small, so CIs are wide.

### Next: `03_optimize.ipynb`
Feed the corrected ε into the margin-maximizing price `p* = c·ε/(ε+1)` and score counterfactual uplift.